[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hany73-max/PDE-Rover-Engine/blob/main/labs_ar/Lab_1_Elimination_of_Constants_AR.ipynb)


In [ ]:
# 🚀 تهيئة بيئة Google Colab
# إذا كنت تشغل هذا النوت بوك على Google Colab، شغل هذه الخلية لتحميل ملفات المستودع والاختبارات تلقائياً:
import os, sys
try:
    if 'google.colab' in str(get_ipython()):
        if not os.path.exists('PDE-Rover-Engine'):
            import subprocess
            subprocess.run(['git', 'clone', 'https://github.com/hany73-max/PDE-Rover-Engine.git'], check=True)
        os.chdir('PDE-Rover-Engine')
        if '.' not in sys.path:
            sys.path.insert(0, '.')
        print('[OK] تم تجهيز بيئة Google Colab بنجاح والاتصال بمستودع المشروع!')
except NameError:
    pass


# 📡 اللاب 1: تكوين المعادلات التفاضلية الجزئية بحذف الثوابت الاختيارية
### *غرفة التحكم: حساس مسح تضاريس المريخ*
**المقرر:** المعادلات التفاضلية الجزئية وتطبيقاتها في الملاحة الفضائية (Applied PDE & Planetary Robotics)  
**موضوع المنهج:** الباب الأول، القسم 1.2 (صفحات 6–11)

---
### 🧭 حجر رشيد (The Rosetta Stone): ترجمة الرياضيات إلى لغة البشر وكود بايثون
| 1. معادلات السبورة في المحاضرة | 2. المعنى الفيزيائي البسيط | 3. كود بايثون المقابل |
| :--- | :--- | :--- |
| $p = \frac{\partial z}{\partial x}, \quad q = \frac{\partial z}{\partial y}$ | "ميل التضاريس وانحدارها باتجاه الشرق وباتجاه الشمال." | `P, Q = np.gradient(Z, dx, dy)` |
| $F(x, y, z, a, b) = 0$ | "عائلة أسطح أو حفر تتغير بتغير البارامترات $a$ و $b$." | `Z = (X - a)**2 + (Y - b)**2` |
| $f(x, y, z, p, q) = 0$ | "القانون الفيزيائي العام الذي يربط الارتفاع بالميول." | `residual = (P**2 + Q**2) - 4*Z` |

---
## 1. المعنى البسيط: "قالب النحات"
تخيل أنك تطبع تضاريس كوكب المريخ ثلاثية الأبعاد. كل حفرة أو كثيب رملي قد يكون له ارتفاع مختلف $a$ وموقع مختلف $b$.
إذا هبطت مركبة الفضاء على هذه التضاريس، فإن نظام الملاحة لا يهتم إطلاقاً بالأرقام $a$ و $b$ المكتوبة على الخريطة؛ كل ما يهم عجلات المركبة في اللحظة الحالية هو:
> *"ما هو انحدار السطح تحت عجلاتي الآن؟ هل الميل شديد جداً بحيث تنقلب المركبة؟"*

**حذف الثوابت (Eliminating Constants)** هو العملية الرياضية لحذف البارامترات الخاصة $a$ و $b$ للوصول إلى **القانون الفيزيائي العام (Universal Invariant Law)** الذي يحكم انحدار السطح في كل نقطة، أياً كان حجم الحفرة أو موقعها!

---
## 2. ما فائدة هذه الطريقة ولماذا نقوم بها؟ (The "WHY")
* **لماذا نقوم بها؟** القوانين الفيزيائية (كالجاذبية والاحتكاك) ثابتة ولا تتغير بتغير تسميات البارامترات $a, b$.
* **ماذا تفيد مركبة المريخ؟** تمكن كمبيوتر الملاحة من فحص أمان المنحدرات مباشرة من قراءات حساسات الميل (IMU Sensors) في الوقت الحقيقي دون الحاجة لخرائط مسبقة!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from labs.tests import test_lab1

%matplotlib inline
print("تم تهيئة بيئة اللاب 1 بنجاح! جاهز لغرفة التحكم.")


---
## 3. الوصفة الرياضية ومثال المذكرة المحلول (Worked Example)

### الحالات الثلاث (صفحة 7 بالمذكرة):
1. **الحالة (1):** عدد الثوابت الاختيارية $<$ عدد المتغيرات المستقلة $\implies$ ينتج أكثر من معادلة من الرتبة الأولى.
2. **الحالة (2):** عدد الثوابت الاختيارية $=$ عدد المتغيرات المستقلة $\implies$ تنتج معادلة واحدة وحيدة من الرتبة الأولى.
3. **الحالة (3):** عدد الثوابت الاختيارية $>$ عدد المتغيرات المستقلة $\implies$ تنتج معادلات من رتب أعلى (Higher Order).

### مثال محلول من مذكرات المحاضرة (صفحة 8):
عائلة الأسطح المكافئة:
$$z = a x + b y + a^2 + b^2$$
* **الخطوة 1:** اشتق جزئياً بالنسبة لـ $x$:  
  $$\frac{\partial z}{\partial x} = p = a$$
* **الخطوة 2:** اشتق جزئياً بالنسبة لـ $y$:  
  $$\frac{\partial z}{\partial y} = q = b$$
* **الخطوة 3:** عوّض بقيمتي $a = p$ و $b = q$ في المعادلة الأصلية:  
  $$z = x p + y q + p^2 + q^2$$
* **النتيجة:** معادلة تفاضلية جزئية غير خطية من الرتبة الأولى خالية تماماً من الثوابت!


---
## 4. 📝 تحدي الورقة والقلم 1 (مسألة من مذكرات المنهج)

> 💡 **تعليمات:** خذ ورقة وقلماً بجانبك، واتبع الخطوات التالية للحل باليد:  
> *(إذا واجهتك أي صعوبة، افتح `INSTRUCTOR_ANSWER_KEY.md` القسم 1.1 لرؤية الخطوات الجبرية كاملة).*

**المسألة:** لدينا عائلة الحفر النيزكية الدائرية:
$$z = (x - a)^2 + (y - b)^2$$
1. اشتق جزئياً بالنسبة لـ $x$: $p = \frac{\partial z}{\partial x} = 2(x - a) \implies (x - a) = \frac{p}{2}$.
2. اشتق جزئياً بالنسبة لـ $y$: $q = \frac{\partial z}{\partial y} = 2(y - b) \implies (y - b) = \frac{q}{2}$.
3. ربع المعادلتين واجمعهما معاً.
4. عوّض بقيمة $z$ من المعادلة الأصلية في الناتج.

اكتب المعادلة الناتجة كنص في الخلية التالية (مثال: `"p^2 + q^2 = 4z"`):


In [ ]:
# خلية تقييم: تحدي الورقة 1
### START CODE HERE ### (≈ 1 line of code)
# اكتب نص المعادلة الناتجة من حلك الورقي (مثال: "p^2 + q^2 = 4z")
student_pde_1 = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab1.check_paper_challenge_1(student_pde_1)


---
## 5. 🚀 تحدي جسر مركبة المريخ (تطبيق المنهج على حوض الحفرة)

الآن نطبق هذا المفهوم مباشرة على **نظام ملاحة مركبة المريخ**:

حوض هبوط المركبة على سطح المريخ يوصف بسطح تربيعي:
$$z = \frac{1}{2} a x^2 + \frac{1}{2} b y^2$$

**على ورقتك:**
1. احسب $p = \frac{\partial z}{\partial x} = a x \implies a = \frac{p}{x}$.
2. احسب $q = \frac{\partial z}{\partial y} = b y \implies b = \frac{q}{y}$.
3. عوّض بقيمتي $a$ و $b$ داخل المعادلة الأصلية ثم اضرب الطرفين في 2.

ما هي المعادلة الناتجة؟ اكتبها أدناه:


In [ ]:
# خلية تقييم: تحدي مركبة المريخ
### START CODE HERE ### (≈ 1 line of code)
# اكتب نص المعادلة الناتجة (مثال: "x*p + y*q = 2z")
rover_basin_pde = None
### END CODE HERE ###

# تشغيل اختبار التحقق التلقائي:
test_lab1.check_paper_challenge_2(rover_basin_pde)


---
## 6. 💻 التطبيق البرمجي ببايثون: التحقق من ميول التضاريس

سنكتب الآن دالة برمجية لكمبيوتر ملاحة المركبة تقوم بفحص شبكة الارتفاعات والتأكد من مطابقتها للمعادلة:
$$\text{residual} = (p^2 + q^2) - 4z \approx 0$$


In [ ]:
# دالة تقييم برمجية: verify_crater_pde
def verify_crater_pde(Z: np.ndarray, P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    # تحسب المتبقي للمعادلة: (p^2 + q^2) - 4*z
    ### START CODE HERE ### (≈ 1 line of code)
    # احسب: (P**2 + Q**2) - 4.0 * Z
    residual = None
    ### END CODE HERE ###
    return residual

# تشغيل الاختبار الآلي:
test_lab1.test_eliminate_constants_fn(verify_crater_pde)


---
## 7. 🎨 الرؤية البصرية للفيزياء: رسم ثلاثي الأبعاد ومجال الميول
شغّل الخلية التالية لرؤية حوض الحفرة ثلاثي الأبعاد مع متجهات الانحدار $(p, q)$ التي ترشد عجلات المركبة:


In [ ]:
x = np.linspace(-3, 3, 40)
y = np.linspace(-3, 3, 40)
X, Y = np.meshgrid(x, y)
Z = (X - 0.5)**2 + (Y + 0.5)**2  # a=0.5, b=-0.5

dx = x[1] - x[0]
dy = y[1] - y[0]
# np.gradient يعيد [dZ/dy, dZ/dx]
Q, P = np.gradient(Z, dy, dx)

fig = plt.figure(figsize=(12, 5))

# رسم ثلاثي الأبعاد لسطح الحفرة
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='copper', alpha=0.85)
ax1.set_title("Mars Crater Basin Surface $z(x, y)$", fontsize=12)

# رسم ثنائي الأبعاد لمجال انحدار الميول
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contour(X, Y, Z, levels=15, cmap='copper')
ax2.quiver(X[::3, ::3], Y[::3, ::3], P[::3, ::3], Q[::3, ::3], color='red')
ax2.set_title("Slope Field $(p, q) = (\\partial z/\\partial x, \\partial z/\\partial y)$", fontsize=12)

plt.tight_layout()
plt.show()

print("تهانينا! لقد أتممت اللاب 1 وأتقنت تكوين المعادلات بحذف الثوابت.")
